In [ ]:
import os

folder_path = r"E:\projects\DTM\Lidar\file"
files = os.listdir(folder_path)

for f in files:
    print(f)


In [ ]:
!pip install requests tqdm


In [ ]:
!pip install requests tqdm


In [ ]:
import os
import requests
from tqdm import tqdm

# 텍스트 파일들이 있는 폴더 경로
input_folder = r"E:\projects\DTM\Lidar\file"

# 해당 폴더의 모든 .txt 파일 경로를 리스트로 생성
txt_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".txt")]

# 파일별 다운로드 함수
def download_file(url, dest_folder):
    filename = url.split("/")[-1]
    filepath = os.path.join(dest_folder, filename)

    if os.path.exists(filepath):
        tqdm.write(f"✅ Skipped (already exists): {filename}")
        return

    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 8192

    if response.status_code == 200:
        with open(filepath, 'wb') as f, tqdm(
            desc=f"⬇️ {filename}",
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            leave=False
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
        tqdm.write(f"✅ Downloaded: {filename}")
    else:
        tqdm.write(f"❌ Failed to download: {filename} (Status code: {response.status_code})")

# 각 텍스트 파일 처리
for txt_path in txt_files:
    # 폴더 이름 생성 (예: FL_PeaceRiver_2014)
    folder_name = os.path.splitext(os.path.basename(txt_path))[0]
    output_dir = os.path.join(r"D:\LiDAR", folder_name)
    os.makedirs(output_dir, exist_ok=True)

    # URL 리스트 읽기
    with open(txt_path, 'r') as f:
        urls = [line.strip() for line in f if line.strip().startswith("http")]

    tqdm.write(f"\n📂 시작: {folder_name} ({len(urls)} files)")
    for url in tqdm(urls, desc=f"Downloading {folder_name}", unit="file"):
        download_file(url, output_dir)
